![Noteable.ac.uk Banner](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/images/Noteable%20NB%20Header%20Banner.png)

## Exemplar Information

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Purpose:</b> This exemplar introduces interactive exoplanet dynamics using the <code>rebound</code> N-body simulator. It demonstrates how compact planetary systems can be modelled and compared through basic orbital setup, resonance-adjacent configurations, time integration, stability diagnostics, energy drift checks, and interactive parameter exploration. It is designed to build intuition about how small changes in eccentricity, mass, and orbital spacing can affect system stability.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Intended audience / teaching context:</b> Designed for students in introductory to intermediate astronomy, physics, or computational science sessions. Suitable for classroom demonstration, guided lab work, or independent practice when introducing numerical simulation, orbital mechanics, resonance, and stability concepts.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Noteable Requirements:</b><br>
    <b>Environment:</b> LIVE 2026<br>
    <b>Server:</b> Astronomy<br>
    <b>Kernel:</b> Python 3
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Required data or dependencies:</b>
    <br>
    - No external dataset required; all planetary system parameters are defined directly in the notebook<br>
    - Python packages: <code>numpy</code>, <code>pandas</code>, <code>matplotlib</code>, <code>plotly</code>, <code>ipywidgets</code>, <code>rebound</code>, <code>IPython</code><br>
    - Standard library modules: <code>warnings</code>
</div>

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Date created / last reviewed:</b> 13 July 2026<br>
    <b>Maintainer / owner:</b> Nik Yusuf
</div>

# When Planetary Systems Go Wrong: Interactive Exoplanet Dynamics, Resonance, and Stability

## Legend

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>blue</b>, the <b>instructions</b> and <b>goals</b> are highlighted. This tells you what we are trying to achieve.
</div>
<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>green</b>, key <b>information</b> and <b>concept explanations</b> are highlighted.
</div>
<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>yellow</b>, <b>exercises</b> and <b>tasks</b> are highlighted for you to try yourself.
</div>
<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>red</b>, <b>error interpretation</b> and <b>debugging tips</b> are highlighted.
</div>

## 1. Setting Up Our Toolkit

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Build a compact exoplanet dynamics lab using <b>rebound</b>. We will create a tightly packed planetary system, simulate its motion, compare a calm and a more troubled architecture, and measure simple stability diagnostics.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Why this matters:</b> Many real exoplanet systems are compact. Small changes in eccentricity, spacing, or planet mass can change whether the system stays orderly or becomes interaction-prone.
</div>

Click the code cell below and press the **<code>▶</code> Run** button in the toolbar at the top (or use `Shift + Enter`).

In [ ]:
# LIVE COMPATIBLE
print('Starting setup: importing simulation, plotting, and widget tools...')

import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import ipywidgets as widgets
import rebound

from IPython.display import display

warnings.filterwarnings('ignore')
plt.style.use('default')
pio.renderers.default = 'notebook_connected'

print('Success! Imports are ready. We can now explore how compact planetary systems behave over time.')

## 2. Designing a Teaching Planetary System

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Define one exoplanet-inspired system that is simple enough to teach but rich enough to show resonance, perturbations, and stability warnings.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>System idea:</b> We will use a Sun-like star with two close-in planets whose orbital periods start close to a <b>3:2</b> ratio. That puts them near a resonance-rich part of parameter space where gravitational interactions matter.
</div>

This is a <b>teaching model inspired by compact exoplanet systems</b>, not a precision fit to one observed system. That makes the notebook fast, robust, and honest about what it can support.

In [ ]:
print('Defining the baseline planetary architecture...')

EARTH_TO_MSUN = 3.003e-6

# EDIT THIS VALUE: mass of the central star in solar masses
stellar_mass_msun = 1.0

# EDIT THIS VALUE: inner planet mass in Earth masses
inner_mass_earth = 8.0

# EDIT THIS VALUE: outer planet mass in Earth masses
outer_mass_earth = 10.0

# EDIT THIS VALUE: inner planet semi-major axis in AU
a_inner_au = 0.10

# EDIT THIS VALUE: target period ratio, close to a 3:2 configuration
baseline_period_ratio = 1.53

# EDIT THIS VALUE: inner planet starting eccentricity
inner_e0 = 0.01

# EDIT THIS VALUE: calm baseline eccentricity for the outer planet
baseline_outer_e0 = 0.02

# EDIT THIS VALUE: deliberately more disturbed comparison value
perturbed_outer_e0 = 0.22

# EDIT THIS VALUE: starting phase offset between the planets in degrees
phase_offset_deg = 180.0

a_outer_au = a_inner_au * baseline_period_ratio ** (2/3)
period_inner_yr = np.sqrt(a_inner_au ** 3 / stellar_mass_msun)
period_outer_yr = np.sqrt(a_outer_au ** 3 / stellar_mass_msun)

system_df = pd.DataFrame([
    {
        'Body': 'Star',
        'Mass': f'{stellar_mass_msun:.2f} solar masses',
        'Semi-major axis (AU)': 'Centre',
        'Initial eccentricity': '-',
        'Orbital period (yr)': '-'
    },
    {
        'Body': 'Inner planet',
        'Mass': f'{inner_mass_earth:.1f} Earth masses',
        'Semi-major axis (AU)': f'{a_inner_au:.3f}',
        'Initial eccentricity': f'{inner_e0:.2f}',
        'Orbital period (yr)': f'{period_inner_yr:.4f}'
    },
    {
        'Body': 'Outer planet',
        'Mass': f'{outer_mass_earth:.1f} Earth masses',
        'Semi-major axis (AU)': f'{a_outer_au:.3f}',
        'Initial eccentricity': f'{baseline_outer_e0:.2f} (baseline) / {perturbed_outer_e0:.2f} (comparison)',
        'Orbital period (yr)': f'{period_outer_yr:.4f}'
    }
])

display(system_df)
print(f'Initial period ratio = {period_outer_yr / period_inner_yr:.3f}')
print('Baseline system defined. We are ready to build the N-body simulation.')

## 3. Building a Simple N-body Model

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Simplifying assumptions:</b> Our model is coplanar, begins from simple orbital elements, and ignores tides, migration, disc physics, and relativity. That is completely acceptable for a teaching notebook, as long as we say so clearly.
</div>

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Create reusable simulation and diagnostic functions so we can compare multiple planetary architectures in a reproducible way.
</div>

Run the next cell once. It creates the main tools used in the rest of the notebook.

In [ ]:
print('Defining helper functions for simulation, integration, and diagnostics...')

def mutual_hill_radius(a1, a2, m1, m2, mstar):
    return ((m1 + m2) / (3 * mstar)) ** (1/3) * ((a1 + a2) / 2)


def build_simulation(outer_e=0.02, outer_mass_scale=1.0, period_ratio=1.53):
    a_outer = a_inner_au * period_ratio ** (2/3)
    m_inner = inner_mass_earth * EARTH_TO_MSUN
    m_outer = outer_mass_earth * outer_mass_scale * EARTH_TO_MSUN

    sim = rebound.Simulation()
    sim.units = ('AU', 'yr', 'Msun')
    sim.G = 4 * np.pi**2
    sim.integrator = 'whfast'

    sim.add(m=stellar_mass_msun)
    sim.add(m=m_inner, a=a_inner_au, e=inner_e0, inc=0.0, Omega=0.0, omega=0.0, M=np.deg2rad(0.0))
    sim.add(m=m_outer, a=a_outer, e=outer_e, inc=0.0, Omega=0.0, omega=0.0, M=np.deg2rad(phase_offset_deg))
    sim.move_to_com()
    sim.dt = np.sqrt(a_inner_au ** 3 / stellar_mass_msun) / 50

    params = {
        'm_star_msun': stellar_mass_msun,
        'm_inner_msun': m_inner,
        'm_outer_msun': m_outer,
        'a_inner_au': a_inner_au,
        'a_outer_au': a_outer,
        'period_ratio_initial': period_ratio,
        'outer_e_initial': outer_e,
        'outer_mass_scale': outer_mass_scale
    }
    return sim, params


def integrate_system(sim, params, t_max_yr=30, n_outputs=500):
    times = np.linspace(0, t_max_yr, n_outputs)
    initial_energy = sim.energy()
    rows = []

    for t in times:
        sim.integrate(t, exact_finish_time=0)
        star = sim.particles[0]
        inner = sim.particles[1]
        outer = sim.particles[2]

        orbit_inner = inner.orbit(primary=star)
        orbit_outer = outer.orbit(primary=star)

        x_inner = inner.x - star.x
        y_inner = inner.y - star.y
        x_outer = outer.x - star.x
        y_outer = outer.y - star.y
        planet_sep = np.sqrt((outer.x - inner.x)**2 + (outer.y - inner.y)**2 + (outer.z - inner.z)**2)

        if orbit_inner.a > 0 and orbit_outer.a > 0:
            period_ratio = (orbit_outer.a / orbit_inner.a) ** 1.5
        else:
            period_ratio = np.nan

        rows.append({
            'time_yr': t,
            'x_inner_au': x_inner,
            'y_inner_au': y_inner,
            'x_outer_au': x_outer,
            'y_outer_au': y_outer,
            'a_inner_au': orbit_inner.a,
            'a_outer_au': orbit_outer.a,
            'e_inner': orbit_inner.e,
            'e_outer': orbit_outer.e,
            'planet_separation_au': planet_sep,
            'period_ratio': period_ratio
        })

    df = pd.DataFrame(rows)
    hill = mutual_hill_radius(
        params['a_inner_au'],
        params['a_outer_au'],
        params['m_inner_msun'],
        params['m_outer_msun'],
        params['m_star_msun']
    )
    final_energy = sim.energy()
    min_sep_hill = df['planet_separation_au'].min() / hill

    summary = {
        'Initial outer eccentricity': params['outer_e_initial'],
        'Outer mass scale': params['outer_mass_scale'],
        'Initial period ratio': params['period_ratio_initial'],
        'Minimum separation (AU)': df['planet_separation_au'].min(),
        'Minimum separation / mutual Hill': min_sep_hill,
        'Max inner eccentricity': df['e_inner'].max(),
        'Max outer eccentricity': df['e_outer'].max(),
        'Mean period ratio': df['period_ratio'].mean(),
        'Std period ratio': df['period_ratio'].std(),
        'Close-approach samples (< 3 Hill)': int((df['planet_separation_au'] < 3 * hill).sum()),
        'Relative energy drift': abs((final_energy - initial_energy) / initial_energy),
        'Mutual Hill radius (AU)': hill,
        'Classroom interpretation': 'stable-looking' if min_sep_hill > 3.5 else 'interaction-prone'
    }
    return df, summary


def run_scenario(name, outer_e, outer_mass_scale=1.0, period_ratio=baseline_period_ratio, t_max_yr=30, n_outputs=500):
    sim, params = build_simulation(outer_e=outer_e, outer_mass_scale=outer_mass_scale, period_ratio=period_ratio)
    df, summary = integrate_system(sim, params, t_max_yr=t_max_yr, n_outputs=n_outputs)
    summary['Scenario'] = name
    return df, summary


print('Helper functions ready. We can now compare different planetary architectures cleanly.')

## 4. Watching the Baseline System Evolve

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Run a baseline near-resonant system and inspect its orbital tracks and time-dependent orbital elements.
</div>

We are asking a simple first question: does this compact system stay reasonably orderly over the time we simulate?

In [ ]:
print('Running the baseline near-resonant simulation...')

# EDIT THIS VALUE: total integration time in years
baseline_duration_yr = 30

# EDIT THIS VALUE: number of stored output times
baseline_samples = 500

baseline_df, baseline_summary = run_scenario(
    name='Baseline near-resonant system',
    outer_e=baseline_outer_e0,
    outer_mass_scale=1.0,
    period_ratio=baseline_period_ratio,
    t_max_yr=baseline_duration_yr,
    n_outputs=baseline_samples
)

# Fixed columns list formatting
columns_to_show = [
    'Scenario',
    'Initial outer eccentricity',
    'Initial period ratio',
    'Minimum separation (AU)',
    'Minimum separation / mutual Hill',
    'Max inner eccentricity',
    'Max outer eccentricity',
    'Std period ratio',
    'Relative energy drift',
    'Classroom interpretation'
]

display(pd.DataFrame([baseline_summary])[columns_to_show].round(5))

print('Baseline integration complete. Next, we will visualise the system.')


In [ ]:
print('Preparing baseline visualisations...')

orbit_limit = 1.15 * np.nanmax(np.abs(baseline_df[['x_inner_au', 'y_inner_au', 'x_outer_au', 'y_outer_au']].to_numpy()))

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=('Orbital tracks in the star frame', 'Eccentricity evolution over time')
)

fig.add_trace(go.Scatter(
    x=baseline_df['x_inner_au'],
    y=baseline_df['y_inner_au'],
    mode='lines',
    line=dict(color='#4C72B0', width=2),
    name='Inner planet'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=baseline_df['x_outer_au'],
    y=baseline_df['y_outer_au'],
    mode='lines',
    line=dict(color='#DD8452', width=2),
    name='Outer planet'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=[0],
    y=[0],
    mode='markers',
    marker=dict(size=14, color='gold', line=dict(color='black', width=1)),
    name='Star'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=baseline_df['time_yr'],
    y=baseline_df['e_inner'],
    mode='lines',
    line=dict(color='#4C72B0', width=2),
    name='Inner eccentricity',
    showlegend=False
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=baseline_df['time_yr'],
    y=baseline_df['e_outer'],
    mode='lines',
    line=dict(color='#DD8452', width=2),
    name='Outer eccentricity',
    showlegend=False
), row=1, col=2)

fig.update_xaxes(title_text='x (AU)', range=[-orbit_limit, orbit_limit], row=1, col=1)
fig.update_yaxes(title_text='y (AU)', range=[-orbit_limit, orbit_limit], scaleanchor='x', scaleratio=1, row=1, col=1)
fig.update_xaxes(title_text='Time (yr)', row=1, col=2)
fig.update_yaxes(title_text='Eccentricity', row=1, col=2)

fig.update_layout(
    template='plotly_white',
    height=500,
    width=1050,
    title='Baseline system: compact, near-resonant, and fairly orderly'
)
fig.show()

print('Baseline visualisation ready. Small oscillations are normal because the planets perturb one another continuously.')

## 5. Stable-looking versus Interaction-prone

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Compare a calmer baseline with a more disturbed system in which the outer planet begins with a larger eccentricity.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Diagnostic idea:</b> We will track minimum planet-planet separation, separation in units of the <b>mutual Hill radius</b>, eccentricity growth, and the period ratio. These are useful warning signs, even though they do not replace a full long-term stability study.
</div>

In [ ]:
print('Running the perturbed comparison scenario...')

perturbed_df, perturbed_summary = run_scenario(
    name='Perturbed outer eccentricity',
    outer_e=perturbed_outer_e0,
    outer_mass_scale=1.0,
    period_ratio=baseline_period_ratio,
    t_max_yr=baseline_duration_yr,
    n_outputs=baseline_samples
)

baseline_df = baseline_df.copy()
perturbed_df = perturbed_df.copy()
baseline_df['sep_in_hill'] = baseline_df['planet_separation_au'] / baseline_summary['Mutual Hill radius (AU)']
perturbed_df['sep_in_hill'] = perturbed_df['planet_separation_au'] / perturbed_summary['Mutual Hill radius (AU)']

comparison_df = pd.DataFrame([baseline_summary, perturbed_summary])[ ['Scenario',
                                                                      'Initial outer eccentricity',
                                                                      'Minimum separation (AU)',
                                                                      'Minimum separation / mutual Hill',
                                                                      'Max inner eccentricity',
                                                                      'Max outer eccentricity',
                                                                      'Mean period ratio',
                                                                      'Std period ratio',
                                                                      'Close-approach samples (< 3 Hill)',
                                                                      'Relative energy drift',
                                                                      'Classroom interpretation'] ]

display(comparison_df.round(5))
print('Comparison run complete. The more eccentric setup should now look visibly less calm.')

In [ ]:
print('Rendering side-by-side orbit geometry and stability diagnostics...')

combo_limit = 1.15 * np.nanmax(np.abs(np.vstack([
    baseline_df[['x_inner_au', 'y_inner_au', 'x_outer_au', 'y_outer_au']].to_numpy(),
    perturbed_df[['x_inner_au', 'y_inner_au', 'x_outer_au', 'y_outer_au']].to_numpy()
])))

fig1 = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=('Baseline near-resonant system', 'Perturbed outer orbit')
)

for col, df in zip([1, 2], [baseline_df, perturbed_df]):
    fig1.add_trace(go.Scatter(
        x=df['x_inner_au'],
        y=df['y_inner_au'],
        mode='lines',
        line=dict(color='#4C72B0', width=2),
        name='Inner planet',
        showlegend=(col == 1)
    ), row=1, col=col)
    fig1.add_trace(go.Scatter(
        x=df['x_outer_au'],
        y=df['y_outer_au'],
        mode='lines',
        line=dict(color='#DD8452', width=2),
        name='Outer planet',
        showlegend=(col == 1)
    ), row=1, col=col)
    fig1.add_trace(go.Scatter(
        x=[0],
        y=[0],
        mode='markers',
        marker=dict(size=14, color='gold', line=dict(color='black', width=1)),
        name='Star',
        showlegend=(col == 1)
    ), row=1, col=col)

fig1.update_xaxes(title_text='x (AU)', range=[-combo_limit, combo_limit], row=1, col=1)
fig1.update_yaxes(title_text='y (AU)', range=[-combo_limit, combo_limit], scaleanchor='x', scaleratio=1, row=1, col=1)
fig1.update_xaxes(title_text='x (AU)', range=[-combo_limit, combo_limit], row=1, col=2)
fig1.update_yaxes(title_text='y (AU)', range=[-combo_limit, combo_limit], scaleanchor='x2', scaleratio=1, row=1, col=2)
fig1.update_layout(template='plotly_white', height=520, width=1050, title='Same compact system, different dynamical mood')
fig1.show()

fig2 = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=(
        'Planet-planet separation relative to the mutual Hill scale',
        'Period ratio compared with the 3:2 value'
    )
)

fig2.add_trace(go.Scatter(
    x=baseline_df['time_yr'],
    y=baseline_df['sep_in_hill'],
    mode='lines',
    line=dict(color='#4C72B0', width=2),
    name='Baseline separation / Hill'
), row=1, col=1)
fig2.add_trace(go.Scatter(
    x=perturbed_df['time_yr'],
    y=perturbed_df['sep_in_hill'],
    mode='lines',
    line=dict(color='#C44E52', width=2),
    name='Perturbed separation / Hill'
), row=1, col=1)

fig2.add_trace(go.Scatter(
    x=baseline_df['time_yr'],
    y=baseline_df['period_ratio'],
    mode='lines',
    line=dict(color='#4C72B0', width=2),
    name='Baseline period ratio'
), row=2, col=1)
fig2.add_trace(go.Scatter(
    x=perturbed_df['time_yr'],
    y=perturbed_df['period_ratio'],
    mode='lines',
    line=dict(color='#C44E52', width=2),
    name='Perturbed period ratio'
), row=2, col=1)

fig2.update_xaxes(title_text='Time (yr)', row=2, col=1)
fig2.update_yaxes(title_text='Separation / mutual Hill', row=1, col=1)
fig2.update_yaxes(title_text='Period ratio', row=2, col=1)
fig2.update_layout(template='plotly_white', height=700, width=980, title='Quantitative diagnostics: why one architecture looks riskier')
fig2.add_hline(y=3.0, line_dash='dash', line_color='black', row=1, col=1)
fig2.add_hline(y=1.5, line_dash='dash', line_color='gray', row=2, col=1)
fig2.show()

print('Diagnostic plots ready. Lower separation margins and stronger period-ratio variations are useful warning signs.')

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Exercise:</b> Go back to the system-definition cell and try changing the baseline period ratio from <b>1.53</b> to <b>1.50</b> or <b>1.58</b>. Then rerun the function cell and the comparison cells. Does the same stability story survive?
</div>

## 6. Interactive Perturbation Lab

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Interactivity:</b> Now you get to change the architecture yourself. Increase the outer eccentricity, make the outer planet heavier, or nudge the period ratio, then see how the safety margin between the planets responds.
</div>

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Try this:</b> Start near the baseline values, then increase the outer eccentricity. After that, try a heavier outer planet. Finally, move the period ratio slightly. Watch how the minimum separation in mutual Hill radii changes.
</div>

Run the next cell, then use the sliders.

In [ ]:
print('Building the interactive perturbation lab...')

def explore_system(outer_e=0.02, outer_mass_scale=1.0, period_ratio=1.53, simulation_years=20):
    trial_df, trial_summary = run_scenario(
        name='Explorer run',
        outer_e=outer_e,
        outer_mass_scale=outer_mass_scale,
        period_ratio=period_ratio,
        t_max_yr=simulation_years,
        n_outputs=320
    )

    hill = trial_summary['Mutual Hill radius (AU)']
    plot_limit = 1.15 * np.nanmax(np.abs(trial_df[['x_inner_au', 'y_inner_au', 'x_outer_au', 'y_outer_au']].to_numpy()))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot(trial_df['x_inner_au'], trial_df['y_inner_au'], color='#4C72B0', lw=1.8, label='Inner planet')
    axes[0].plot(trial_df['x_outer_au'], trial_df['y_outer_au'], color='#DD8452', lw=1.8, label='Outer planet')
    axes[0].scatter([0], [0], s=120, color='gold', edgecolor='black', zorder=5, label='Star')
    axes[0].set_xlim(-plot_limit, plot_limit)
    axes[0].set_ylim(-plot_limit, plot_limit)
    axes[0].set_aspect('equal', adjustable='box')
    axes[0].set_title('Orbital tracks')
    axes[0].set_xlabel('x (AU)')
    axes[0].set_ylabel('y (AU)')
    axes[0].legend(loc='upper right')

    axes[1].plot(trial_df['time_yr'], trial_df['planet_separation_au'] / hill, color='purple', lw=2)
    axes[1].axhline(3.0, color='black', linestyle='--', linewidth=1, label='3 mutual Hill radii')
    axes[1].set_title('Separation safety margin')
    axes[1].set_xlabel('Time (yr)')
    axes[1].set_ylabel('Separation / mutual Hill')
    axes[1].legend(loc='upper right')

    plt.tight_layout()
    plt.show()

    print(f"Detected behaviour: {trial_summary['Classroom interpretation']}")
    print(f"Minimum separation / mutual Hill = {trial_summary['Minimum separation / mutual Hill']:.2f}")
    print(f"Maximum outer eccentricity reached = {trial_summary['Max outer eccentricity']:.3f}")
    print(f"Mean period ratio during the run = {trial_summary['Mean period ratio']:.3f}")
    print(f"Relative energy drift = {trial_summary['Relative energy drift']:.2e}")


widgets.interact(
    explore_system,
    outer_e=widgets.FloatSlider(
        value=baseline_outer_e0,
        min=0.00,
        max=0.26,
        step=0.02,
        description='Outer e:',
        continuous_update=False
    ),
    outer_mass_scale=widgets.FloatSlider(
        value=1.0,
        min=0.5,
        max=3.0,
        step=0.25,
        description='Mass scale:',
        continuous_update=False
    ),
    period_ratio=widgets.FloatSlider(
        value=baseline_period_ratio,
        min=1.48,
        max=1.60,
        step=0.01,
        description='P ratio:',
        continuous_update=False
    ),
    simulation_years=widgets.IntSlider(
        value=20,
        min=10,
        max=40,
        step=5,
        description='Years:',
        continuous_update=False
    )
)

print('Interactive explorer ready. Move one control at a time so the physical effect is easier to interpret.')

## 7. Responsible Interpretation and Model Limits

<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Important caution:</b> A short teaching simulation is not the same as a full stability paper. This notebook is designed to build intuition, not to claim the true long-term fate of a specific observed exoplanet system.
</div>

Here are the main limits to keep in mind:

- <b>Short timescale:</b> a system that looks calm for a few decades may still become unstable over much longer times.
- <b>Simplified architecture:</b> we used only two planets, coplanar motion, and simple starting angles.
- <b>No extra physics:</b> tides, migration, disc forces, and relativity are ignored.
- <b>Teaching parameters:</b> this system is exoplanet-inspired, but not a fitted real system.
- <b>Warning metrics are not magic rules:</b> the mutual Hill scale is useful, but it is not a complete yes-or-no stability test.
- <b>Numerical settings matter:</b> timestep, integration length, and initial conditions all influence what we see.

That is good scientific practice: simulations are powerful, but their meaning always depends on their assumptions.

## Take-away

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    Congratulations! You have built and explored a compact exoplanet system with <b>rebound</b>. You defined a near-resonant architecture, integrated its motion, compared a calmer and a more disturbed configuration, and used quantitative diagnostics to interpret what you saw.
</div>

Strong next steps could include:
- adding a third planet to create a resonant chain
- testing mass perturbations more systematically
- increasing the integration length to see whether the same trends persist
- tracking angular momentum and energy behaviour in more detail
- comparing this teaching system with parameters from a published exoplanet system

This is a great example of modern computational astronomy: a small change in planetary architecture can produce a visibly different dynamical story.

![Noteable license](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/images/Noteable%20Notebook%20Footer.png)